# Multi-slice DLPFC Integration

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import torch

from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

import SpaDiff as sd
from SpaDiff.utils import mclust_R, set_seed

## Configuration

In [ ]:
SEED = 42
ST_SAMPLES = ["151673", "151674", "151675", "151676"]
# ST_SAMPLES = ["151669", "151670", "151671", "151672"]
# ST_SAMPLES = ["151507", "151508", "151509", "151510"]
BATCH_KEY = "batch_name"
N_CLUSTERS = 7

DATA_ROOT = Path("E:/gxy_2/final/0_data/case1")
print("DATA_ROOT =", DATA_ROOT)

set_seed(SEED)
torch.backends.cudnn.deterministic = True
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device =", device)

## Data loading

In [ ]:
slices = []
for sample in ST_SAMPLES:
    sample_dir = DATA_ROOT / sample
    current = sc.read_visium(sample_dir)
    current.var_names_make_unique()
    current.layers["counts"] = current.X.copy()
    sc.pp.normalize_total(current, target_sum=1e4)
    sc.pp.log1p(current)

    current, _ = sd.spatial_reconstruction(current, alpha=1.5)

    truth = pd.read_csv(sample_dir / "truth.txt", sep="\t", header=None, index_col=0)
    truth.columns = ["Truth"]
    current.obs["Truth"] = truth.reindex(current.obs_names)["Truth"]
    current.obs[BATCH_KEY] = sample
    current.obs_names = [f"{sample}:{barcode}" for barcode in current.obs_names]
    slices.append(current)

adata = sc.concat(slices, join="inner", merge="same")

sc.pp.highly_variable_genes(
    adata,
    flavor="seurat_v3",
    layer="counts",
    n_top_genes=3000,
    batch_key=BATCH_KEY,
    subset=True,
)


adata

## Higher-order topology

In [ ]:
topology = sd.build_spatial_topology(
    adata,
    batch_key=BATCH_KEY,
    slice_order=ST_SAMPLES,
    device=device,
)
operators = topology.operators

sc.tl.pca(adata, n_comps=50)
features = torch.as_tensor(adata.obsm["X_pca"], dtype=torch.float32, device=device)

In [ ]:
print("features:", tuple(features.shape), "topology mode:", topology.mode)
print("simplex counts:", topology.simplex_counts)
print("operator nnz by order:", {order: operator._nnz() for order, operator in operators.items()})


## Batch-conditioned VP-SDE training

In [ ]:
config = sd.SpaDiffConfig(
    num_batches=len(ST_SAMPLES),
)
model = sd.SpaDiff(config).to(device)
adata = model.fit_transform(
    adata,
    features,
    operators,
    batch_key=BATCH_KEY,
    batch_order=ST_SAMPLES,
)

In [ ]:
labels = mclust_R(
    adata, num_cluster=N_CLUSTERS, used_obsm="X_spadiff", pca_num=20
)
adata.obs["mclust"] = pd.Categorical(labels.astype(str))

ari_by_slice = {}
nmi_by_slice = {}

for sample in ST_SAMPLES:
    current = adata.obs.loc[adata.obs[BATCH_KEY] == sample, ["Truth", "mclust"]].dropna()
    ari_by_slice[sample] = adjusted_rand_score(current["Truth"], current["mclust"])
    nmi_by_slice[sample] = normalized_mutual_info_score(current["Truth"], current["mclust"])
valid = adata.obs[["Truth", "mclust"]].dropna()

overall_ari = round(adjusted_rand_score(valid["Truth"], valid["mclust"]), 4)
overall_nmi = round(normalized_mutual_info_score(valid["Truth"], valid["mclust"]), 4)

print("ARI by slice:", {key: round(value, 3) for key, value in ari_by_slice.items()})
print("NMI by slice:", {key: round(value, 3) for key, value in nmi_by_slice.items()})

print(f"overall ARI = {overall_ari:.3f}")
print(f"overall NMI = {overall_nmi:.3f}")


In [ ]:
palette = ["#6D1A9C", "#D1D1D1", "#F56867", "#59BE86", "#FEB915", "#C798EE", "#7495D3"]
fig, axes = plt.subplots(1, len(ST_SAMPLES), figsize=(5 * len(ST_SAMPLES), 5))
for axis, sample in zip(axes, ST_SAMPLES):
    current = adata[adata.obs[BATCH_KEY] == sample].copy()
    sc.pl.spatial(
        current, color="mclust", ax=axis, show=False, spot_size=120,
        palette=palette, legend_loc=None, title=f"{sample} | ARI={ari_by_slice[sample]:.3f}",
    )
fig.suptitle(f"overall ARI={overall_ari:.3f}", fontsize=16)
plt.tight_layout()
# plt.savefig("../result/spadiff_multi_slice_" + str(overall_ari) + ".pdf", bbox_inches="tight")

In [ ]:
# output_file = "../result/spadiff_multi_slice_" + str(overall_ari) + ".h5ad"

# adata.write_h5ad(output_file, compression="gzip")